NER MODEL

MEDAES QA

In [ ]:
# Script to extract questions from the 'medaesqa_v1.json' dataset
import json

def get_json():
    try:
        with open('datasets/medaesqa_v1.json', 'r', encoding="utf-8") as file:
            print("Successfully loaded 'medaesqa_v1.json'.")
            return json.load(file)
    except FileNotFoundError:
        print("File not found. Please ensure 'medaesqa_v1.json' is in the current directory.")
        return None
    except json.JSONDecodeError:
        print("Error decoding JSON. Please ensure 'medaesqa_v1.json' contains valid JSON.")
        return None
    

def extract_questions(data):
    questions = []

    if data:
        for item in data:
            if "question" in item:
                questions.append(item["question"])
    else:
        print("No data found.")

    return questions



data = get_json()
questions = extract_questions(data)

for q in questions:
    print(q)

In [ ]:
# Script to convert extracted questions to csv file
import csv
def save_to_csv(questions, filename='extracted_questions.csv'):
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Question'])  # Write header
            for question in questions:
                writer.writerow([question])  # Write each question in a new row
        print(f"Questions successfully saved to '{filename}'.")
    except Exception as e:
        print(f"Error occurred while saving to CSV: {e}")  
        
save_to_csv(questions)

In [86]:
import re

def classify_medical_intent(query):
    query = query.lower()
    
    # Define keywords for each intent
    intent_map = {
        "SIDE_EFFECTS": ["side effect", "complication", "what effects", "suddenly stopping"],
        "DRUG_INTERACTION": ["interaction", "combination of drugs", "can i use", "after using", "with", "mix"],
        "ALCOHOL_INTERACTION": ["alcohol", "drinking", "beer", "wine"],
        "FOOD_INTERACTION": ["food", "diet", "ketogenic", "eat", "grapefruit"],
        "USES_INDICATIONS": ["used for", "how to treat", "treatment for", "treatment options", "what can be done", "how to manage", "interventions"],
        "STEWARDSHIP_MISSED": ["don't take", "missed", "forgot", "if i stop"],
        "REDIRECT_MEDICINE_QUERY": ["cause", "prevent", "natural", "heal", "how long", "why do i", "symptoms", "outcome", "test to rule out"],
        "MEDICAL_DEFINITION": ["what is", "what does", "mean", "definition", "mutation in"],
        "STEWARDSHIP_ADHERENCE": ["how long after", "keep dropping", "finish the course"],
        "REDIRECT_DOSAGE_QUERY": ["dosage", "mg", "dose", "how much", "how many"],
        "WARNING_PRECAUTIONS": ["warning", "caution", "safe to", "danger"],
        "ANTIBIOTIC_INFO": ["antibiotic", "penicillin", "amoxicillin"],
        "ABOUT_CHATBOT": ["who are you", "what can you do"],
        "DICTIONARY_LOOKUP": ["spelling", "pronounce"]
    }

    # Check for keyword matches
    for intent, keywords in intent_map.items():
        if any(keyword in query for keyword in keywords):
            return intent
            
    return "NOT_RECOGNIZED"

# Example usage with your provided list
questions = [
    "What are side effects of using formoterol?",
    "what is tamsulosin used for?",
    "can i use Sudafed after using Afrin for nasal congestion?",
    "what does cad without angina mean?"
]

for q in questions:
    print(f"Query: {q}\nIntent: {classify_medical_intent(q)}\n")

Query: What are side effects of using formoterol?
Intent: SIDE_EFFECTS

Query: what is tamsulosin used for?
Intent: USES_INDICATIONS

Query: can i use Sudafed after using Afrin for nasal congestion?
Intent: DRUG_INTERACTION

Query: what does cad without angina mean?
Intent: DRUG_INTERACTION



Medical Info 2019

In [ ]:
pip install pandas

In [91]:
import pandas as pd

# Load dataset
med_info_raw_data = pd.read_csv("datasets/MedInfo2019-QA-Medications.csv")

# Drop unnecessary columns
med_info_raw_data.drop(['URL', 'Section Title', 'Answer'], axis=1, inplace=True)

# Your list of antibiotics to keep
antibiotics = [
    "Amoxicillin",
    "Clarithromycin",
    "Azithromycin",
    "Doxycycline",
    "Cefuroxime",
    "Levofloxacin",
    "Cefixime",
    "Cefotaxime",
    "Fosfomycin",
    "Nitrofurantoin",
    "Trimethoprim-sulfamethoxazole",
    "Ciprofloxacin"
]

# List of types to remove
remove_types = [
    "Action",
    "Dose",
    "Ingredient",
    "Action/time",
    "Comparison",
    "Stopping/tapering",
    "not_drug_question",
    "Alternatives",
    "Availability",
    "Pronounce name",
    "Dose/Potency",
    "Time/duration",
    "Action/effectiveness",
    "other_drug_question",
    "Usage/time"
]

# Step 1: Automatically keep rows that mention your antibiotics
pattern = "|".join(antibiotics)
keep_antibiotics = med_info_raw_data[
    med_info_raw_data['Focus (Drug)'].str.contains(pattern, case=False, na=False)
]

# Step 2: From the remaining rows, remove unwanted types
filtered_rest = med_info_raw_data[
    ~med_info_raw_data['Focus (Drug)'].str.contains(pattern, case=False, na=False)  # not antibiotics
]
filtered_rest = filtered_rest[~filtered_rest['Question Type'].isin(remove_types)]

# Step 3: Combine the two sets
med_info_final = pd.concat([keep_antibiotics, filtered_rest]).reset_index(drop=True)

# Check
print(med_info_final.head())
print(f"Original rows: {len(med_info_raw_data)}, Final rows: {len(med_info_final)}")

filtered_rest.to_csv("checking.csv")

                                            Question       Focus (Drug)  \
0         can i take metamucil with "ciprofloxacin?"      ciprofloxacin   
1                  can i take tea with azithromycin?       azithromycin   
2  how long i take cipro for a urinary tract infe...      ciprofloxacin   
3    how long does it take cefuroxime axetil to work  Cefuroxime Axetil   
4               who makes this drug nitrofurantoin ?     Nitrofurantoin   

   Question Type  
0    Interaction  
1    Interaction  
2     Usage/time  
3  Time/duration  
4   Manufacturer  
Original rows: 690, Final rows: 440


In [93]:
import pandas as pd
import re
import random
from collections import defaultdict

# Load your dataset
df = pd.read_csv('checking.csv')

# Your target drug list (the ones you want to KEEP)
target_drugs = [
    'Amoxicillin',
    'Clarithromycin', 
    'Azithromycin', 
    'Doxycycline',
    'Cefuroxime',
    'Levofloxacin',
    'Cefixime',
    'Cefotaxime',
    'Fosfomycin',
    'Nitrofurantoin',
    'Trimethoprim-sulfamethoxazole',
    'Ciprofloxacin'
]

# Create a normalized version for comparison (lowercase)
target_drugs_lower = [drug.lower() for drug in target_drugs]

# Identify your columns
question_col = 'Question'  # Your question column name
drug_col = 'Focus (Drug)'  # Your drug column name

# Set random seed for reproducibility (optional)
random.seed(42)

# Track counts to maintain balance
replacement_counts = defaultdict(int)

def map_to_target_drug_balanced(original_drug):
    """Map unknown drugs to maintain balanced distribution"""
    original_lower = str(original_drug).lower().strip()
    
    # Skip unwanted terms
    skip_terms = ['marijuana', 'cannabis', 'calcium', 'vitamin', 'herbal', 'supplement']
    if any(term in original_lower for term in skip_terms):
        return None
    
    # Find which target drug has been used least for replacements
    if not replacement_counts:
        # If no replacements yet, initialize all counts to 0
        for drug in target_drugs:
            replacement_counts[drug] = 0
    
    min_count = min(replacement_counts.values())
    least_used = [drug for drug in target_drugs if replacement_counts[drug] == min_count]
    
    # Randomly choose among the least used
    chosen = random.choice(least_used)
    replacement_counts[chosen] += 1
    
    return chosen

# Transform the dataset
transformed_rows = []
stats = {
    'kept_original': 0,
    'replaced': 0,
    'skipped': 0,
    'skipped_terms': 0
}

for idx, row in df.iterrows():
    # Get the original values
    original_question = str(row[question_col])
    original_drug = str(row[drug_col]).strip()
    
    # Skip if drug is empty
    if not original_drug or original_drug.lower() == 'nan':
        stats['skipped'] += 1
        continue
    
    # Check if the drug is already in your target list
    original_drug_lower = original_drug.lower()
    
    # Find if this drug matches any in your target list
    is_in_target = False
    matched_target = None
    
    # First, check for exact matches
    if original_drug_lower in target_drugs_lower:
        is_in_target = True
        matched_target = target_drugs[target_drugs_lower.index(original_drug_lower)]
    else:
        # Check if drug name contains any target drug or vice versa
        for target_drug, target_lower in zip(target_drugs, target_drugs_lower):
            if target_lower in original_drug_lower or original_drug_lower in target_lower:
                is_in_target = True
                matched_target = target_drug
                break
    
    if is_in_target:
        # Drug is already in your list - KEEP IT
        if original_drug != matched_target:
            # Replace with correct capitalization if needed
            pattern = re.compile(re.escape(original_drug), re.IGNORECASE)
            new_question = pattern.sub(matched_target, original_question)
            new_drug = matched_target
        else:
            # Already perfect, keep as is
            new_question = original_question
            new_drug = original_drug
        stats['kept_original'] += 1
    else:
        # Drug is NOT in your list - REPLACE IT with balanced random
        target_drug = map_to_target_drug_balanced(original_drug)
        
        # Check if drug was skipped due to skip terms
        if target_drug is None:
            stats['skipped_terms'] += 1
            continue
        
        # Replace the drug name in the question
        pattern = re.compile(re.escape(original_drug), re.IGNORECASE)
        new_question = pattern.sub(target_drug, original_question)
        new_drug = target_drug
        stats['replaced'] += 1
    
    # Create transformed row
    new_row = row.to_dict()
    new_row[question_col] = new_question
    new_row[drug_col] = new_drug
    new_row['original_drug'] = original_drug  # Keep for reference
    
    transformed_rows.append(new_row)

# Create new dataframe
new_df = pd.DataFrame(transformed_rows)

# Print detailed statistics
print("\n" + "="*60)
print("TRANSFORMATION STATISTICS")
print("="*60)
print(f"Total rows processed: {len(df)}")
print(f"✅ Kept original drugs (already in list): {stats['kept_original']}")
print(f"🔄 Replaced drugs (not in list): {stats['replaced']}")
print(f"⚠️ Skipped (empty drug): {stats['skipped']}")
print(f"⚠️ Skipped (unwanted terms like marijuana, etc): {stats['skipped_terms']}")
print(f"\nFinal dataset size: {len(new_df)}")

# Show distribution of drugs in the transformed dataset
print("\n" + "="*60)
print("DRUG DISTRIBUTION IN TRANSFORMED DATASET")
print("="*60)
drug_counts = new_df[drug_col].value_counts()
for drug, count in drug_counts.items():
    percentage = (count / len(new_df)) * 100
    print(f"{drug}: {count} ({percentage:.1f}%)")

# Show replacement distribution (how many times each drug was used for replacements)
if stats['replaced'] > 0:
    print("\n" + "="*60)
    print("REPLACEMENT DISTRIBUTION (How many unknown drugs mapped to each target)")
    print("="*60)
    for drug, count in replacement_counts.items():
        percentage = (count / stats['replaced']) * 100 if stats['replaced'] > 0 else 0
        print(f"{drug}: {count} ({percentage:.1f}%)")

# Show samples of both kept and replaced drugs
print("\n" + "="*60)
print("SAMPLE TRANSFORMATIONS")
print("="*60)

# Show examples of kept drugs
kept_samples = new_df[new_df['original_drug'] == new_df[drug_col]].head(3)
if len(kept_samples) > 0:
    print("\n✅ KEPT (drug was already in your list):")
    for _, row in kept_samples.iterrows():
        print(f"  Original: {row['original_drug']} → Kept as: {row[drug_col]}")
        print(f"  Question: {row[question_col][:80]}...")
        print()

# Show examples of replaced drugs
replaced_samples = new_df[new_df['original_drug'] != new_df[drug_col]].head(5)
if len(replaced_samples) > 0:
    print("\n🔄 REPLACED (drug was NOT in your list):")
    for _, row in replaced_samples.iterrows():
        print(f"  Original: {row['original_drug']} → Replaced with: {row[drug_col]}")
        print(f"  Question: {row[question_col][:80]}...")
        print()

# Save transformed dataset
output_file = 'transformed_dataset.csv'
new_df.to_csv(output_file, index=False)
print(f"\n✅ Transformed dataset saved to '{output_file}'")

# Optional: Save a mapping log for reference
mapping_log = new_df[['original_drug', drug_col]].drop_duplicates()
mapping_log.to_csv('drug_mapping_log.csv', index=False)
print(f"📝 Mapping log saved to 'drug_mapping_log.csv'")


TRANSFORMATION STATISTICS
Total rows processed: 425
✅ Kept original drugs (already in list): 0
🔄 Replaced drugs (not in list): 402
⚠️ Skipped (empty drug): 0
⚠️ Skipped (unwanted terms like marijuana, etc): 23

Final dataset size: 402

DRUG DISTRIBUTION IN TRANSFORMED DATASET
Clarithromycin: 34 (8.5%)
Amoxicillin: 34 (8.5%)
Cefuroxime: 34 (8.5%)
Azithromycin: 34 (8.5%)
Ciprofloxacin: 34 (8.5%)
Fosfomycin: 34 (8.5%)
Trimethoprim-sulfamethoxazole: 33 (8.2%)
Cefixime: 33 (8.2%)
Levofloxacin: 33 (8.2%)
Doxycycline: 33 (8.2%)
Cefotaxime: 33 (8.2%)
Nitrofurantoin: 33 (8.2%)

REPLACEMENT DISTRIBUTION (How many unknown drugs mapped to each target)
Amoxicillin: 34 (8.5%)
Clarithromycin: 34 (8.5%)
Azithromycin: 34 (8.5%)
Doxycycline: 33 (8.2%)
Cefuroxime: 34 (8.5%)
Levofloxacin: 33 (8.2%)
Cefixime: 33 (8.2%)
Cefotaxime: 33 (8.2%)
Fosfomycin: 34 (8.5%)
Nitrofurantoin: 33 (8.2%)
Trimethoprim-sulfamethoxazole: 33 (8.2%)
Ciprofloxacin: 34 (8.5%)

SAMPLE TRANSFORMATIONS

🔄 REPLACED (drug was NOT in 